# EXP-004 — Đối chứng λ = 0 (không neo), seed 2 và 3, 30 epoch
Bổ sung để so sánh cơ chế đủ 3 seed. λ = 0 seed 1 = EXP-003. Cấu hình y hệt λ = 1 (cache, lr 1e-3, 30 epoch), chỉ khác trọng số neo = 0.
Vẫn truyền ngân hàng text để **ghi log loss neo** (không ảnh hưởng train), xem head trôi bao xa khỏi CLIP.

So sánh ở epoch 30, trung bình 3 seed: λ = 0 vs λ = 1 vs EXP-000 (5 epoch gốc).

# Cấu hình

In [1]:
SEEDS = [2, 3]
EPOCHS = 30
SAVE_EPOCHS = [30]                    # chỉ test mốc cuối
LR = 1e-3
LAMBDAS = [0]                         # đối chứng: không neo
ANCHOR = "cross"                      # cross: KL ảnh→text bank | text: MSE ma trận cos giữa các câu
BANK_VARIANT = "A"                    # A: không tên lớp | B: thêm từ cảm xúc (ablation)
job_of = lambda lam, seed: f"exp004_anchor_{ANCHOR}{BANK_VARIANT}_l{lam:g}_l14_seed{seed}"
BRANCH = "exp/004-text-anchor-loss"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"     # target đã cache ở EXP-001 (targets/chunk_*.npz)
CLIP_CACHE = f"{PRE_DIR}/clip_feats_l14.npz"                       # feature CLIP, tính 1 lần
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
BANK = f"{OUT_DIR}/text_bank_{BANK_VARIANT}_l14.npz"
L1_CSVS = [f"{OUT_DIR}/exp004_anchor_{ANCHOR}{BANK_VARIANT}_l14_seed1-results.csv",      # λ = 1 seed 1
           f"{OUT_DIR}/exp004_anchor_{ANCHOR}{BANK_VARIANT}_l14_seed2_3-results.csv"]    # λ = 1 seed 2, 3
EXP003_CSV = "/content/drive/MyDrive/ExpCLIP_output/exp003/exp003_cache_l14_seed1-results.csv"   # λ = 0 seed 1

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

# Mount Drive & GPU

In [3]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
# không nạp BLIP-2 → GPU nào cũng được (T4/L4/A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


# Clone repo (nhánh EXP-004)

In [4]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 409 (delta 299), reused 375 (delta 266), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 32.24 MiB | 13.56 MiB/s, done.
Resolving deltas: 100% (299/299), done.
/content/Exp-CLIP
Already on 'exp/004-text-anchor-loss'
Your branch is up to date with 'origin/exp/004-text-anchor-loss'.
Already up to date.
119131b (HEAD -> exp/004-text-anchor-loss, origin/exp/004-text-anchor-loss) EXP-004: add executed anchor-loss notebook (lambda 1, seeds 2 and 3)


# Giải nén dữ liệu (train + test)

In [5]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [6]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# Cache feature CLIP + ngân hàng text
Đã có trên Drive từ seed 1 thì bỏ qua.

In [7]:
if not os.path.exists(CLIP_CACHE):
    run(f'/content/venv310/bin/python tools/cache_clip_features.py --train-data {TRAIN_DATA} --out {CLIP_CACHE} '
        f'--load-model CLIP_L14 --batch-size 256 --workers 8')
if not os.path.exists(BANK):
    run(f'/content/venv310/bin/python tools/build_text_bank.py --variant {BANK_VARIANT} --out {BANK} --load-model CLIP_L14')
!ls -lh "$CLIP_CACHE" "$BANK"

-rw------- 1 root root  75M Oct 14 07:10 /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
-rw------- 1 root root 576K Oct 14 15:16 /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz


# Train λ = 0 (seed 2, 3; 30 epoch, từ cache)
Bị ngắt thì chạy lại cell này: phần đã xong được bỏ qua, phần đang dở tự resume.

In [8]:
for seed in SEEDS:
  for lam in LAMBDAS:
    job = job_of(lam, seed)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-model.pth"):
        print(f"seed {seed}, λ={lam}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== seed {seed}, λ = {lam}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {seed} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
        f'--cache-clip {CLIP_CACHE} --cache-targets {PRE_DIR}/targets '
        f'--text-bank {BANK} --anchor-weight {lam} --anchor-type {ANCHOR} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

=============== seed 2, λ = 0
Training date:  26-10-15 12:56
************************
workers = 8
epochs = 30
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 1000
milestones = [1000]
seed = 2
job_id = exp004_anchor_crossA_l0_l14_seed2
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp004
train_data = /data/EECS-IoannisLab/datasets/Static_FER_Datasets/CAERS_Face/train
sample_weights = None
cache_clip = /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
cache_targets = /content/drive/MyDrive/ExpCLIP_output/exp001/full/targets
text_bank = /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz
anchor_weight = 0.0
anchor_type = cross
save_epochs = [30]
resume = None
************************

  0%|        

# Test zero-shot từng mốc

In [9]:
!mkdir -p checkpoint log
for seed in SEEDS:
  for lam in LAMBDAS:
    for e in SAVE_EPOCHS:
        job = f"{job_of(lam, seed)}-e{e}"
        if os.path.exists(f"{OUT_DIR}/{job}-test.txt"):
            !cp "$OUT_DIR/$job-test.txt" log/
            continue
        if not os.path.exists(f"checkpoint/{job}-model.pth"):
            !cp "$OUT_DIR/$job-model.pth" checkpoint/
        print(f"=============== seed {seed}, λ = {lam}, epoch {e}")
        run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {job} '
            f'--data-root /content/static_test_sets 2>&1 | tee log/{job}-test.txt')
        !cp log/$job-test.txt "$OUT_DIR/"

=============== seed 2, λ = 0, epoch 30
************************
load_model = CLIP_L14
job_id = exp004_anchor_crossA_l0_l14_seed2-e30
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 52.21/62.13
************************* AffectNet7
UAR/WAR: 42.94/42.94
************************* AffectNet8
UAR/WAR: 36.92/36.92
************************* FERPlus
UAR/WAR: 42.76/45.40
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 52.64/63.23
************************* AffectNet7
UAR/WAR: 43.49/43.49
************************* AffectNet8
UAR/WAR: 37.55/37.55
************************* FER

# So sánh (epoch 30, trung bình ± std 3 seed)
Bảng 1: Ensemble Five từng seed. Bảng 2: Ensemble Five, λ = 0 vs λ = 1, Δ so với EXP-000 mean3. Bảng 3: đủ 5 prompt, Δ so với EXP-000 mean3.

In [10]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

def parse(path):
    rows, prompt, dataset = [], None, None
    for line in open(path):
        if "Prompt Type:" in line:
            prompt = line.split("Prompt Type:")[1].strip()
        elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
            dataset = line.strip().split()[-1]
        elif line.startswith("UAR/WAR:"):
            uar, war = map(float, line.split(":")[1].strip().split("/"))
            rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
    return pd.DataFrame(rows)

res = pd.concat([parse(f"log/{job_of(lam, seed)}-e{e}-test.txt").assign(seed=seed, lam=lam, epoch=e)
                 for seed in SEEDS for lam in LAMBDAS for e in SAVE_EPOCHS])
res.to_csv(f"{OUT_DIR}/exp004_anchor_{ANCHOR}{BANK_VARIANT}_l0_l14_seed{'_'.join(map(str, SEEDS))}-results.csv", index=False)
r0 = pd.read_csv(EXP003_CSV).assign(seed=1, lam=0)
l1 = pd.concat([pd.read_csv(f) for f in L1_CSVS])
l1 = l1.assign(seed=l1.get("seed", pd.Series(1, index=l1.index)).fillna(1).astype(int))
res = pd.concat([r0, res, l1[l1.lam == 1]])
res = res[res.epoch == 30]
assert res.groupby("lam").seed.nunique().eq(3).all(), res.groupby("lam").seed.unique()
pd.set_option("display.width", 250)
DS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
E5 = "Expression Ensemble Five"
m3 = REF[REF.src == "EXP-000 mean3"].set_index(["prompt", "dataset"]).UAR

ens = res[res.prompt == E5]
display(ens.pivot_table(index="lam", columns=["dataset", "seed"], values="UAR")[DS])

g = ens.groupby(["lam", "dataset"]).UAR
mean, std = g.mean().unstack()[DS], g.std().unstack()[DS]
display(pd.concat({"mean": mean.round(2), "std": std.round(2), "Δ vs EXP-000 mean3": (mean - m3.loc[E5]).round(2)}, axis=1))

full = res.groupby(["lam", "prompt", "dataset"]).UAR.agg(["mean", "std"]).reset_index()
full = full.merge(m3.rename("UAR_base").reset_index(), on=["prompt", "dataset"])
full["Δ"] = (full["mean"] - full.UAR_base).round(2)
display(full.pivot_table(index=["prompt", "dataset"], columns="lam", values="Δ", sort=False))

dataset  RAFDB               AffectNet7               AffectNet8               FERPlus              
seed         1      2      3          1      2      3          1      2      3       1      2      3
lam                                                                                                 
0.0      53.68  52.66  55.75      44.20  43.51  43.94      37.85  37.65  37.00   44.96  43.46  45.36
1.0      58.73  57.07  58.26      45.83  43.97  45.46      39.60  38.75  39.07   46.14  45.63  45.34

mean                                 std                               Δ vs EXP-000 mean3                              
dataset  RAFDB AffectNet7 AffectNet8 FERPlus RAFDB AffectNet7 AffectNet8 FERPlus              RAFDB AffectNet7 AffectNet8 FERPlus
lam                                                                                                                              
0.0      54.03      43.88      37.50   44.59  1.57       0.35       0.44    1.00              -4.04      -0.29      -1.34    3.04
1.0      58.02      45.09      39.14   45.70  0.86       0.98       0.43    0.41              -0.05       0.92       0.30    4.15

lam                                                       0.0   1.0
prompt                                       dataset               
A Photo of A Face with An Expression of Name AffectNet7 -1.67 -0.37
                                             AffectNet8 -2.27 -0.79
                                             FERPlus     3.79  5.93
                                             RAFDB      -4.40  3.13
An Expression of Name                        AffectNet7  0.28  0.54
                                             AffectNet8 -0.73  0.75
                                             FERPlus     7.75  6.79
                                             RAFDB      -1.78  1.52
Class Name                                   AffectNet7  2.23  0.37
                                             AffectNet8  1.10  0.32
                                             FERPlus     5.16  4.39
                                             RAFDB       2.78  3.38
Expression Ensemble Five                     AffectNet7 -0.29  0.92
                                             AffectNet8 -1.34  0.30
                                             FERPlus     3.04  4.15
                                             RAFDB      -4.04 -0.05
Expression Ensemble Ten                      AffectNet7 -0.91  0.12
                                             AffectNet8 -1.66 -0.30
                                             FERPlus     2.12  3.90
                                             RAFDB      -4.26  1.25

# Log train
Loss neo (chỉ ghi log, λ = 0): head trôi khỏi CLIP bao xa. λ = 1 kết thúc ở ~0.05.

In [11]:
import re
for seed in SEEDS:
  for lam in LAMBDAS:
    lines = open(f"{OUT_DIR}/{job_of(lam, seed)}-log.txt").read().splitlines()
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    anc = [float(re.search(r"Anchor [\d.]+ \(([\d.]+)\)", l).group(1)) for l in lines if "Anchor" in l]
    print(f"seed {seed}, λ={lam}: acc", {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)},
          "| anchor", [round(a, 3) for a in anc[::max(1, len(anc) // 6)]])

seed 2, λ=0: acc {30: 81.27} | anchor [0.297, 2.966, 2.91, 3.122, 3.279, 3.506]
seed 3, λ=0: acc {30: 81.16} | anchor [0.322, 3.149, 3.02, 3.107, 3.337, 3.544]
